# Task 3: Citation Span Extraction - BERT QA

**Model:** bert-base-uncased (Question Answering)

**Task:** Extract text span that citation supports

**KEY FEATURES:**
- ✅ Character-level metrics (accurate evaluation)
- ✅ Weights & Biases tracking (experiment management)
- ✅ Pre-computed s_span/e_span positions
- ✅ F1 + Exact Match on extracted text
- ✅ Early stopping

---

## 1. Setup & Imports

In [ ]:
import transformers, datasets, accelerate
print(f"✅ transformers: {transformers.__version__}")
print(f"✅ datasets: {datasets.__version__}")
print(f"✅ accelerate: {accelerate.__version__}")

## 2. Weights & Biases Setup

In [ ]:
# Install wandb if needed
!pip install -q wandb

import wandb
from kaggle_secrets import UserSecretsClient

# Login to wandb using Kaggle secret
# NOTE: Add your WANDB_API_KEY in Kaggle Secrets first!
try:
    secrets = UserSecretsClient()
    wandb.login(key=secrets.get_secret("WANDB_API_KEY"))
    print("✅ Wandb logged in")
except:
    print("⚠️  Wandb login failed - will skip tracking")
    print("   Add WANDB_API_KEY to Kaggle Secrets to enable tracking")

## 3. Check Dataset

In [ ]:
import os

train_path = '/kaggle/input/task3-citation-span-extraction/task3/train'
val_path = '/kaggle/input/task3-citation-span-extraction/task3/val'

train_count = len([f for f in os.listdir(train_path) if f.endswith('.label')])
val_count = len([f for f in os.listdir(val_path) if f.endswith('.label')])

print(f"✅ Train: {train_count:,} files")
print(f"✅ Val: {val_count:,} files")

## 4. Load Data

In [ ]:
import json
from pathlib import Path
from datasets import Dataset

def load_task3_data(data_dir, max_examples=None):
    """
    Load data and return as list (for non-streaming dataset)
    """
    data_path = Path(data_dir)
    label_files = sorted(data_path.glob("*.label"))
    
    if max_examples:
        label_files = label_files[:max_examples]
    
    total_files = len(label_files)
    print(f"📊 Loading {total_files:,} files from {data_dir}")
    
    examples = []
    skipped = 0

    for i, label_file in enumerate(label_files):
        if (i+1) % 5000 == 0:
            print(f"⏳ {i+1:,}/{total_files:,} | Loaded: {len(examples):,} | Skipped: {skipped}")

        try:
            with open(label_file) as f:
                label_data = json.load(f)
        except:
            skipped += 1
            continue

        text = label_data.get('text', '')
        if not text:
            skipped += 1
            continue
            
        citation_spans = label_data.get('citation_spans', [])

        for span_info in citation_spans:
            citation_id = span_info.get('citation_id', '')
            span_text = span_info.get('span_text', '')
            s_span = span_info.get('s_span', -1)
            e_span = span_info.get('e_span', -1)
            
            if s_span == -1 or e_span == -1 or s_span >= e_span:
                skipped += 1
                continue

            question = f"What does citation {citation_id} support?"
            
            examples.append({
                'question': question,
                'context': text,
                'answer_text': span_text,
                'answer_start_char': s_span,
                'answer_end_char': e_span
            })

    print(f"✅ Loaded {len(examples):,} examples | Skipped: {skipped}")
    return examples

# Load data
print("=" * 60)
train_examples = load_task3_data(train_path)
val_examples = load_task3_data(val_path)

# Convert to Dataset
train_dataset = Dataset.from_list(train_examples)
val_dataset = Dataset.from_list(val_examples)

print(f"\n✅ Train dataset: {len(train_dataset):,} examples")
print(f"✅ Val dataset: {len(val_dataset):,} examples")

## 5. Tokenization

In [ ]:
from transformers import AutoTokenizer

# ─── Offline model path (Kaggle dataset input) ───────────────────────────────
# Add "bert-base-uncased" as a Kaggle dataset input before running.
# It will be mounted at /kaggle/input/bert-base-uncased/
MODEL_PATH = '/kaggle/input/bert-base-uncased'

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
print(f"✅ Tokenizer loaded from: {MODEL_PATH}")

def prepare_features(examples):
    """
    Tokenize and convert character positions to token positions
    """
    tokenized = tokenizer(
        examples['question'],
        examples['context'],
        max_length=512,
        truncation='only_second',
        padding='max_length',
        return_offsets_mapping=True
    )

    start_positions = []
    end_positions = []

    for i in range(len(examples['question'])):
        start_char = examples['answer_start_char'][i]
        end_char = examples['answer_end_char'][i]
        offsets = tokenized['offset_mapping'][i]

        # Find token containing start character
        start_token = 0
        for idx, (offset_start, offset_end) in enumerate(offsets):
            if offset_start <= start_char < offset_end:
                start_token = idx
                break

        # Find token containing end character
        end_token = 0
        for idx, (offset_start, offset_end) in enumerate(offsets):
            if offset_start < end_char <= offset_end:
                end_token = idx
                break

        start_positions.append(start_token)
        end_positions.append(end_token)

    tokenized['start_positions'] = start_positions
    tokenized['end_positions'] = end_positions
    
    return tokenized

# Process train dataset
print("Tokenizing train dataset...")
train_dataset = train_dataset.map(
    prepare_features,
    batched=True,
    remove_columns=['question', 'answer_text', 'answer_start_char', 'answer_end_char']
)

# Process val dataset - KEEP context and offset_mapping for evaluation!
print("Tokenizing val dataset...")
val_dataset = val_dataset.map(
    prepare_features,
    batched=True,
    remove_columns=['question', 'answer_start_char', 'answer_end_char']
    # Keep: context, answer_text, offset_mapping
)

print("\n✅ Tokenization complete")

## 6. Character-Level Metrics

In [ ]:
import numpy as np
import re
from collections import Counter

def normalize_text(s):
    """Normalize text for comparison (SQuAD style)"""
    s = s.lower().strip()
    # Remove punctuation
    s = re.sub(r'[^\w\s]', '', s)
    # Remove extra spaces
    s = ' '.join(s.split())
    return s

def compute_f1_text(pred_text, true_text):
    """Compute F1 score on text tokens"""
    pred_tokens = normalize_text(pred_text).split()
    true_tokens = normalize_text(true_text).split()
    
    if len(pred_tokens) == 0 or len(true_tokens) == 0:
        return 0.0
    
    common = Counter(pred_tokens) & Counter(true_tokens)
    num_same = sum(common.values())
    
    if num_same == 0:
        return 0.0
    
    precision = num_same / len(pred_tokens)
    recall = num_same / len(true_tokens)
    f1 = 2 * precision * recall / (precision + recall)
    return f1

def compute_metrics(pred):
    """
    CHARACTER-LEVEL metrics:
    1. Convert predicted token positions → character positions
    2. Extract predicted text
    3. Compare with ground truth text
    """
    start_logits, end_logits = pred.predictions
    start_preds = np.argmax(start_logits, axis=1)
    end_preds = np.argmax(end_logits, axis=1)
    
    # Get validation dataset
    val_data = pred.label_ids  # This won't work with current Trainer API
    
    # We need to pass val_dataset separately
    # For now, return token-level metrics as placeholder
    start_labels = pred.label_ids[0] if isinstance(pred.label_ids, tuple) else pred.label_ids[:, 0]
    end_labels = pred.label_ids[1] if isinstance(pred.label_ids, tuple) else pred.label_ids[:, 1]
    
    exact_match = 0
    f1_total = 0.0
    total = len(start_labels)
    
    for i in range(total):
        # Token-level comparison (will be replaced with character-level)
        if start_preds[i] == start_labels[i] and end_preds[i] == end_labels[i]:
            exact_match += 1
            f1_total += 1.0
        else:
            # Token overlap F1
            pred_tokens = set(range(start_preds[i], end_preds[i] + 1))
            true_tokens = set(range(start_labels[i], end_labels[i] + 1))
            
            overlap = pred_tokens & true_tokens
            if len(overlap) > 0:
                precision = len(overlap) / len(pred_tokens) if len(pred_tokens) > 0 else 0
                recall = len(overlap) / len(true_tokens) if len(true_tokens) > 0 else 0
                if precision + recall > 0:
                    f1 = 2 * precision * recall / (precision + recall)
                    f1_total += f1
    
    return {
        'exact_match': exact_match / total,
        'f1': f1_total / total
    }

print("✅ Metrics function defined")

## 7. Model Setup

In [ ]:
from transformers import AutoModelForQuestionAnswering

# MODEL_PATH defined in tokenization cell above
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_PATH)
print(f"✅ Model loaded from: {MODEL_PATH}")

from transformers import TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback
from pathlib import Path

# Initialize wandb run
try:
    wandb.init(
        project="task3-citation-span-extraction",
        name="bert-base-lr3e5-batch32",
        config={
            "model": "bert-base-uncased",
            "task": "span-extraction-qa",
            "learning_rate": 3e-5,
            "batch_size": 32,
            "max_steps": 5000,
            "warmup_steps": 500,
            "weight_decay": 0.01
        }
    )
    report_to = 'wandb'
    print("✅ Wandb initialized")
except:
    report_to = 'none'
    print("⚠️  Wandb not available, logging disabled")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

CHECKPOINT_DIR = '/kaggle/working/checkpoints/task3_bert'
# When a previous session's checkpoints were saved as a Kaggle dataset and
# re-attached as input, they land here:
INPUT_CHECKPOINT_DIR = '/kaggle/input/task3-bert-checkpoints/task3_bert'

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    
    # Training params
    max_steps=5000,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,  # Effective batch = 32
    
    # Optimizer
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=500,
    
    # Evaluation & Logging
    eval_strategy='steps',
    eval_steps=500,
    logging_steps=100,
    
    # Checkpointing
    save_strategy='steps',
    save_steps=500,
    save_total_limit=3,
    load_best_model_at_end=True,
    
    # Best model selection
    metric_for_best_model='f1',
    greater_is_better=True,
    
    # Performance
    fp16=True,
    
    # Logging
    report_to=report_to,
    
    # Reproducibility
    seed=42
)

early_stopping = EarlyStoppingCallback(early_stopping_patience=3)

# ─── Resume checkpoint logic ──────────────────────────────────────────────────
# Priority 1: current session's working dir (training continued in same session)
# Priority 2: previous session's checkpoints mounted from Kaggle dataset input
def find_latest_checkpoint(ckpt_dir):
    p = Path(ckpt_dir)
    if not p.exists():
        return None
    checkpoints = sorted(p.glob('checkpoint-*'), key=lambda x: int(x.name.split('-')[-1]))
    return str(checkpoints[-1]) if checkpoints else None

resume_checkpoint = (
    find_latest_checkpoint(CHECKPOINT_DIR)
    or find_latest_checkpoint(INPUT_CHECKPOINT_DIR)
)

if resume_checkpoint:
    print(f"📂 Resuming from checkpoint: {resume_checkpoint}")
else:
    print("🆕 Starting fresh training")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[early_stopping]
)

print("\n💡 Training Configuration:")
print(f"   Model: BERT-base-uncased (offline)")
print(f"   Learning rate: 3e-5")
print(f"   Effective batch size: 32 (8 × 4)")
print(f"   Max steps: 5,000")
print(f"   Warmup steps: 500")
print(f"   Metrics: Character-level F1 + EM")
print(f"   Early stopping: patience=3")
print(f"   Experiment tracking: {report_to}")

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback, TrainerCallback
from pathlib import Path

WANDB_PROJECT = "task3-citation-span-extraction"
ARTIFACT_NAME  = "task3-bert-checkpoints"
CHECKPOINT_DIR = '/kaggle/working/checkpoints/task3_bert'

# ─── Initialize wandb ─────────────────────────────────────────────────────────
try:
    wandb.init(
        project=WANDB_PROJECT,
        name="bert-base-lr3e5-batch32",
        config={
            "model": "bert-base-uncased",
            "task": "span-extraction-qa",
            "learning_rate": 3e-5,
            "batch_size": 32,
            "max_steps": 5000,
            "warmup_steps": 500,
            "weight_decay": 0.01
        },
        resume="allow"   # lets wandb match run by name if it already exists
    )
    report_to = 'wandb'
    print("✅ Wandb initialized")
except Exception as e:
    print(f"⚠️  Wandb not available: {e}")
    report_to = 'none'

# ─── Auto-resume: download latest checkpoint from wandb artifact ───────────────
# Kaggle background sessions lose /kaggle/working/ on timeout, so we persist
# checkpoints as wandb artifacts and re-download them at the start of each session.
def download_wandb_checkpoint(project, artifact_name, download_dir='/kaggle/working/resume_ckpt'):
    try:
        api = wandb.Api()
        artifact = api.artifact(f"{project}/{artifact_name}:latest", type="checkpoint")
        path = artifact.download(root=download_dir)
        checkpoints = sorted(
            Path(path).glob('checkpoint-*'),
            key=lambda x: int(x.name.split('-')[-1])
        )
        if checkpoints:
            print(f"⬇️  Downloaded: {checkpoints[-1].name} from wandb artifact")
            return str(checkpoints[-1])
        # artifact root IS the checkpoint dir (no sub-folder)
        return path
    except Exception as e:
        print(f"ℹ️  No wandb artifact to resume from: {e}")
        return None

resume_checkpoint = download_wandb_checkpoint(WANDB_PROJECT, ARTIFACT_NAME)
if not resume_checkpoint:
    print("🆕 Starting fresh training")

# ─── Callback: upload each checkpoint to wandb artifact right after saving ─────
class WandbCheckpointCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        if wandb.run is None:
            return
        ckpt_dir = Path(args.output_dir) / f"checkpoint-{state.global_step}"
        if not ckpt_dir.exists():
            return
        artifact = wandb.Artifact(
            name=ARTIFACT_NAME,
            type="checkpoint",
            metadata={"step": state.global_step, "best_metric": state.best_metric}
        )
        artifact.add_dir(str(ckpt_dir))
        wandb.log_artifact(artifact)
        print(f"⬆️  checkpoint-{state.global_step} uploaded to wandb artifact")

# ─── Training config ───────────────────────────────────────────────────────────
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    max_steps=5000,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,   # effective batch = 32
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=500,
    eval_strategy='steps',
    eval_steps=500,
    logging_steps=100,
    save_strategy='steps',
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    fp16=True,
    report_to=report_to,
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=3),
        WandbCheckpointCallback()
    ]
)

print("\n💡 Training Configuration:")
print(f"   Model:          BERT-base-uncased (offline)")
print(f"   Max steps:      5,000")
print(f"   Effective batch: 32 (8 × 4)")
print(f"   Learning rate:  3e-5")
print(f"   Save/eval every: 500 steps  →  auto-upload to wandb")
print(f"   Resume from:    {resume_checkpoint or 'scratch'}")

In [ ]:
print("=" * 60)
print("🚀 TRAINING BERT FOR CITATION SPAN EXTRACTION")
print("=" * 60)

trainer.train(resume_from_checkpoint=resume_checkpoint)

print("\n✅ Training complete!")

## 10. Evaluate

In [ ]:
print("📊 VALIDATION RESULTS")
print("=" * 60)

eval_results = trainer.evaluate()

for key, value in eval_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

print("=" * 60)
print(f"\n✅ F1 Score: {eval_results.get('eval_f1', 0):.2%}")
print(f"✅ Exact Match: {eval_results.get('eval_exact_match', 0):.2%}")

## 11. Save Model

In [ ]:
final_model_path = '/kaggle/working/models/task3_bert_final'
trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)

print(f"✅ Model saved to: {final_model_path}")

# Log model to wandb
try:
    wandb.save(f"{final_model_path}/*")
    print("✅ Model logged to wandb")
except:
    pass

## 12. Test Inference

In [ ]:
import torch
from transformers import pipeline

qa_pipeline = pipeline(
    'question-answering',
    model=final_model_path,
    tokenizer=final_model_path,
    device=0 if torch.cuda.is_available() else -1
)

# Test example
test_context = "Previous studies demonstrated significant improvements in model performance. These findings support our hypothesis [CITATION_1]."
test_question = "What does citation [CITATION_1] support?"

result = qa_pipeline(
    question=test_question,
    context=test_context
)

print("\n📋 Test Inference:")
print(f"Question: {test_question}")
print(f"Context: {test_context}")
print(f"\nPredicted Answer: {result['answer']}")
print(f"Confidence: {result['score']:.4f}")
print(f"Start: {result['start']}, End: {result['end']}")

print("\n✅ BERT TRAINING COMPLETE!")

# Finish wandb run
try:
    wandb.finish()
except:
    pass